# Azure AI Evaluation client library for Python

Use Azure AI Evaluation SDK to assess the performance of your generative AI applications. Generative AI application generations are quantitatively measured with mathematical based metrics, AI-assisted quality and safety metrics. Metrics are defined as evaluators. Built-in or custom evaluators can provide comprehensive insights into the application's capabilities and limitations.

Use [Azure AI Evaluation SDK](https://pypi.org/project/azure-ai-evaluation/) to:

- Evaluate existing data from generative AI applications
- Evaluate generative AI applications
- Evaluate by generating mathematical, AI-assisted quality and safety metrics

Azure AI SDK provides following to evaluate Generative AI Applications:

- Evaluators - Generate scores individually or when used together with evaluate API.
- Evaluate API - Python API to evaluate dataset or application using built-in or custom evaluators.

In [1]:
!pip install azure-ai-evaluation --quiet

In [2]:
# Fix this
# [INFO] Could not import AIAgentConverter. Please install the dependency with `pip install azure-ai-projects`.
!pip install azure-ai-projects --quiet

---

## 1. Built-in Evaluators

Built-in evaluators are out of box evaluators provided by Microsoft:

| Category |	Evaluator class |
|:---|:--|
| Performance and quality (AI-assisted) |	GroundednessEvaluator, RelevanceEvaluator, CoherenceEvaluator, FluencyEvaluator, SimilarityEvaluator, RetrievalEvaluator |
| Performance and quality (NLP)| 	F1ScoreEvaluator, RougeScoreEvaluator, GleuScoreEvaluator, BleuScoreEvaluator, MeteorScoreEvaluator| 
| Risk and safety (AI-assisted)	| ViolenceEvaluator, SexualEvaluator, SelfHarmEvaluator, HateUnfairnessEvaluator, IndirectAttackEvaluator, ProtectedMaterialEvaluator| 
| Composite	| QAEvaluator, ContentSafetyEvaluator| 

For more in-depth information on each evaluator definition and how it's calculated, [see Evaluation and monitoring metrics for generative AI.](https://learn.microsoft.com/azure/ai-studio/concepts/evaluation-metrics-built-in)

In [3]:
import os

# Project Connection String
connection_string = os.environ.get("AZURE_AI_CONNECTION_STRING")

# Extract details
region_id, subscription_id, resource_group_name, project_name = connection_string.split(";")

# Populate it
azure_ai_project = {
    "subscription_id": subscription_id,
    "resource_group_name": resource_group_name,
    "project_name": project_name,
}
print(azure_ai_project)

{'subscription_id': '0bcab828-911a-4d6e-a39d-92edbc4e2798', 'resource_group_name': 'rg-aitour', 'project_name': 'ai-project-51054003'}


In [4]:
from azure.ai.evaluation import BleuScoreEvaluator

# NLP bleu score evaluator
bleu_score_evaluator = BleuScoreEvaluator()
result = bleu_score_evaluator(
    response="Tokyo is the capital of Japan.",
    ground_truth="The capital of Japan is Tokyo."
)
print(result)

{'bleu_score': 0.22961813530951883, 'bleu_result': 'fail', 'bleu_threshold': 0.5}


In [5]:

from azure.ai.evaluation import RelevanceEvaluator

# AI assisted quality evaluator
model_config = {
    "azure_endpoint": os.environ.get("AZURE_OPENAI_ENDPOINT"),
    "api_key": os.environ.get("AZURE_OPENAI_API_KEY"),
    "azure_deployment": os.environ.get("AZURE_OPENAI_DEPLOYMENT"),
}

relevance_evaluator = RelevanceEvaluator(model_config)
result = relevance_evaluator(
    query="What is the capital of Japan?",
    response="The capital of Japan is Tokyo."
)
print(result)


{'relevance': 5.0, 'gpt_relevance': 5.0, 'relevance_reason': 'The RESPONSE accurately and completely answers the QUERY, providing the correct information without any errors or omissions. Therefore, it deserves the highest relevance score.', 'relevance_result': 'pass', 'relevance_threshold': 3}


In [6]:
from azure.ai.evaluation import ViolenceEvaluator

# AI assisted safety evaluator
violence_evaluator = ViolenceEvaluator(azure_ai_project)
result = violence_evaluator(
    query="What is the capital of France?",
    response="Paris."
)

Class ViolenceEvaluator: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.


TypeError: ViolenceEvaluator.__init__() missing 1 required positional argument: 'azure_ai_project'

---

## 2. Custom Evaluators
Built-in evaluators are great out of the box to start evaluating your application's generations. However you can build your own code-based or prompt-based evaluator to cater to your specific evaluation needs.

In [7]:
# Custom evaluator as a function to calculate response length
def response_length(response, **kwargs):
    return len(response)

# Custom class based evaluator to check for blocked words
class BlocklistEvaluator:
    def __init__(self, blocklist):
        self._blocklist = blocklist

    def __call__(self, *, answer: str, **kwargs):
        contains_block_word = any(word in answer for word in self._blocklist)
        return {"score": contains_block_word}

blocklist_evaluator = BlocklistEvaluator(blocklist=["bad", "worst", "terrible"])

# Test custom evaluator 1
result = response_length("The capital of Japan is Tokyo.")
print(result)

# Test custom evaluator 2
result = blocklist_evaluator(answer="The capital of Japan is Tokyo.")
print(result)

# Test custom evaluator 3
result = blocklist_evaluator(answer="This is a bad idea.")
print(result)

30
{'score': False}
{'score': True}


---

## 3. Evaluate API


### 3.1 Evaluate existing dataset

The package provides an evaluate API which can be used to run multiple evaluators together to evaluate generative AI application response.

For more details refer to [Evaluate on test dataset using evaluate()](https://learn.microsoft.com/azure/ai-studio/how-to/develop/evaluate-sdk#evaluate-on-test-dataset-using-evaluate)

In [ ]:
from azure.ai.evaluation import evaluate

data="data.jsonl", # provide your data here

result = evaluate(
    data="data.jsonl", # provide your data here
    evaluators={
        "blocklist": blocklist_evaluator,
        "relevance": relevance_evaluator
    },
    # column mapping
    evaluator_config={
        "relevance": {
            "column_mapping": {
                "query": "${data.query}",
                "ground_truth": "${data.ground_truth}",
                "response": "${data.answer}"
            } 
        }
    },
    # Optionally provide your AI Foundry project information to track your evaluation results in your Azure AI Foundry project
    azure_ai_project = azure_ai_project,
    # Optionally provide an output path to dump a json of metric summary, row level data and metric and AI Foundry URL
    output_path="./evaluation_results.json"
)

[2025-05-07 16:37:45 +0000][promptflow._core.entry_meta_generator][WARNING] - Generate meta in current process and timeout won't take effect. Please handle timeout manually outside current process.
[2025-05-07 16:37:45 +0000][promptflow._core.entry_meta_generator][WARNING] - Generate meta in current process and timeout won't take effect. Please handle timeout manually outside current process.
[2025-05-07 16:37:45 +0000][promptflow._sdk._orchestrator.run_submitter][INFO] - Submitting run azure_ai_evaluation_evaluators_relevance_20250507_163745_836822, log path: /home/vscode/.promptflow/.runs/azure_ai_evaluation_evaluators_relevance_20250507_163745_836822/logs.txt
[2025-05-07 16:37:45 +0000][promptflow._sdk._orchestrator.run_submitter][INFO] - Submitting run azure_ai_evaluation_evaluators_blocklist_20250507_163745_836222, log path: /home/vscode/.promptflow/.runs/azure_ai_evaluation_evaluators_blocklist_20250507_163745_836222/logs.txt
Traceback (most recent call last):
  File "<string>", 

2025-05-07 16:37:46 +0000   26510 execution.bulk     INFO     Current thread is not main thread, skip signal handler registration in BatchEngine.
2025-05-07 16:37:47 +0000   26510 execution.bulk     INFO     Finished 1 / 3 lines.
2025-05-07 16:37:47 +0000   26510 execution.bulk     INFO     Average execution time for completed lines: 1.91 seconds. Estimated time for incomplete lines: 3.82 seconds.
2025-05-07 16:37:48 +0000   26510 execution.bulk     INFO     Finished 2 / 3 lines.
2025-05-07 16:37:48 +0000   26510 execution.bulk     INFO     Average execution time for completed lines: 1.01 seconds. Estimated time for incomplete lines: 1.01 seconds.
2025-05-07 16:37:48 +0000   26510 execution.bulk     INFO     Finished 3 / 3 lines.
2025-05-07 16:37:48 +0000   26510 execution.bulk     INFO     Average execution time for completed lines: 0.76 seconds. Estimated time for incomplete lines: 0.0 seconds.
======= Run Summary =======

Run name: "azure_ai_evaluation_evaluators_relevance_20250507_

EvaluationException: (InternalError) Failed to start spawned fork process manager

### 3.2 Evaluate generative AI application

Above code snippet refers to askwiki application in this [sample](https://github.com/Azure-Samples/azureai-samples/tree/main/scenarios/evaluate/Supported_Evaluation_Targets/Evaluate_App_Endpoint).

For more details refer to [Evaluate on a target](https://learn.microsoft.com/azure/ai-studio/how-to/develop/evaluate-sdk#evaluate-on-a-target)

In [9]:
from askwiki import askwiki

result = evaluate(
    data="data.jsonl",
    target=askwiki,
    evaluators={
        "relevance": relevance_eval
    },
    evaluator_config={
        "default": {
            "column_mapping": {
                "query": "${data.queries}"
                "context": "${outputs.context}"
                "response": "${outputs.response}"
            } 
        }
    }
)


SyntaxError: invalid syntax (614966383.py, line 13)

## 4. Simulator

Simulators allow users to generate synthentic data using their application. Simulator expects the user to have a callback method that invokes their AI application. The intergration between your AI application and the simulator happens at the callback method. Here's how a sample callback would look like:

In [10]:
async def callback(
    messages: Dict[str, List[Dict]],
    stream: bool = False,
    session_state: Any = None,
    context: Optional[Dict[str, Any]] = None,
) -> dict:
    messages_list = messages["messages"]
    # Get the last message from the user
    latest_message = messages_list[-1]
    query = latest_message["content"]
    # Call your endpoint or AI application here
    # response should be a string
    response = call_to_your_application(query, messages_list, context)
    formatted_response = {
        "content": response,
        "role": "assistant",
        "context": "",
    }
    messages["messages"].append(formatted_response)
    return {"messages": messages["messages"], "stream": stream, "session_state": session_state, "context": context}

NameError: name 'Dict' is not defined

The simulator initialization and invocation looks like this:

In [11]:

from azure.ai.evaluation.simulator import Simulator
model_config = {
    "azure_endpoint": os.environ.get("AZURE_ENDPOINT"),
    "azure_deployment": os.environ.get("AZURE_DEPLOYMENT_NAME"),
    "api_version": os.environ.get("AZURE_API_VERSION"),
}
custom_simulator = Simulator(model_config=model_config)
outputs = asyncio.run(custom_simulator(
    target=callback,
    conversation_turns=[
        [
            "What should I know about the public gardens in the US?",
        ],
        [
            "How do I simulate data against LLMs",
        ],
    ],
    max_conversation_turns=2,
))
with open("simulator_output.jsonl", "w") as f:
    for output in outputs:
        f.write(output.to_eval_qr_json_lines())



ValueError: The following keys in model_config must not be None: azure_deployment, azure_endpoint

### 4.1 Adversarial Simulator

In [12]:
from azure.ai.evaluation.simulator import AdversarialSimulator, AdversarialScenario
from azure.identity import DefaultAzureCredential
azure_ai_project = {
    "subscription_id": <subscription_id>,
    "resource_group_name": <resource_group_name>,
    "project_name": <project_name>
}
scenario = AdversarialScenario.ADVERSARIAL_QA
simulator = AdversarialSimulator(azure_ai_project=azure_ai_project, credential=DefaultAzureCredential())

outputs = asyncio.run(
    simulator(
        scenario=scenario,
        max_conversation_turns=1,
        max_simulation_results=3,
        target=callback
    )
)

print(outputs.to_eval_qr_json_lines())

SyntaxError: invalid syntax (1908608512.py, line 4)